In [67]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [68]:
from pathlib import Path

project_root = Path("/content/drive/MyDrive/music-analytics")

folders = [
    project_root / "app",
    project_root / "uploads",
    project_root / "outputs"
]

for folder in folders:
    folder.mkdir(parents=True, exist_ok=True)

files = [
    project_root / "app" / "preprocess.py",
    project_root / "app" / "spotify_api.py",
    # project_root / "app" / "audio_features.py",
    project_root / "app" / "genre_encoding.py",
    project_root / "app" / "analytics.py",
    project_root / "app" / "pipeline.py",
    project_root / "app" / "generate_track_artists_ids.py",
    project_root / "app" / "generate_audio_genres_features.py",
    project_root / "main.py",
    project_root / "requirements.txt",
    project_root / ".env"
]

for file in files:
    file.touch(exist_ok=True)

print("Project structure created!")

Project structure created!


In [69]:
!find /content/drive/MyDrive/music-analytics

/content/drive/MyDrive/music-analytics
/content/drive/MyDrive/music-analytics/app
/content/drive/MyDrive/music-analytics/app/audio_features.py
/content/drive/MyDrive/music-analytics/app/track_artists_ids.py
/content/drive/MyDrive/music-analytics/app/__pycache__
/content/drive/MyDrive/music-analytics/app/__pycache__/genre_encoding.cpython-312.pyc
/content/drive/MyDrive/music-analytics/app/__pycache__/analytics.cpython-312.pyc
/content/drive/MyDrive/music-analytics/app/__pycache__/generate_track_artists_ids.cpython-312.pyc
/content/drive/MyDrive/music-analytics/app/__pycache__/preprocess.cpython-312.pyc
/content/drive/MyDrive/music-analytics/app/__pycache__/spotify_api.cpython-312.pyc
/content/drive/MyDrive/music-analytics/app/__pycache__/generate_audio_genres_features.cpython-312.pyc
/content/drive/MyDrive/music-analytics/app/generate_track_artists_ids.py
/content/drive/MyDrive/music-analytics/app/spotify_api.py
/content/drive/MyDrive/music-analytics/app/preprocess.py
/content/drive/MyD

In [70]:
import sys

PROJECT_ROOT = "/content/drive/MyDrive/music-analytics"
sys.path.append(PROJECT_ROOT)

In [71]:
# with open(".env", "w") as f:
#     f.write("CLIENT_ID=2ace2638f0c049ee8c50d9bd32a50f0d\n")
#     f.write("CLIENT_SECRET=8ce36a8d743b4361af6e6acaeccfb971\n")
#     f.write("RAPIDAPI_KEY=b0b0f01094msh1ff7875fd311552p1fb67cjsne7ef7645cb0c\n")

In [72]:
%%writefile /content/drive/MyDrive/music-analytics/app/preprocess.py

def load_streaming_history(json_file):
  import pandas as pd
  import numpy as np

  df = pd.read_json(json_file)


  # =========================================
  # CREATE YYYY-MM COLUMN
  # =========================================

  # Convert endTime to datetime
  df["endTime"] = pd.to_datetime(df["endTime"])

  # Extract only year-month
  df["year_month"] = df["endTime"].dt.strftime("%Y-%m")
  return df

def create_monthly_summary(json_file):
  df=load_streaming_history(json_file)
  summary = (
    df.groupby(
        ["year_month", "artistName", "trackName"]
    )
    .agg(
        playCount=("trackName", "count"),
        totalMsPlayed=("msPlayed", "sum")
    )
    .reset_index()
  )

  summary.to_csv(
    "spotify_summary.csv",
    index=False
  )
  return summary

Overwriting /content/drive/MyDrive/music-analytics/app/preprocess.py


In [88]:
%%writefile /content/drive/MyDrive/music-analytics/app/spotify_api.py

import pandas as pd
import time
import base64
import requests

# =========================
# GET ACCESS TOKEN
# =========================

def get_token(CLIENT_ID, CLIENT_SECRET):

    auth_string = f"{CLIENT_ID}:{CLIENT_SECRET}"

    auth_base64 = base64.b64encode(
        auth_string.encode()
    ).decode()

    url = "https://accounts.spotify.com/api/token"

    headers = {
        "Authorization": f"Basic {auth_base64}",
        "Content-Type": "application/x-www-form-urlencoded"
    }

    data = {
        "grant_type": "client_credentials"
    }

    result = requests.post(
        url,
        headers=headers,
        data=data
    )


    # result = requests.post(
    # url,
    # headers=headers,
    # data=data
    # )

    print("Status Code:", result.status_code)
    print("Response:", result.text)

    token = result.json()["access_token"]

    return token


def get_track_and_artist_id(song, artist, token):

    url = "https://api.spotify.com/v1/search"

    headers = {
        "Authorization": f"Bearer {token}"
    }

    query = f"track:{song} artist:{artist}"

    params = {
        "q": query,
        "type": "track",
        "limit": 1
    }

    response = requests.get(
        url,
        headers=headers,
        params=params
    )

    # ERROR CHECK
    if response.status_code != 200:
        print("ERROR:", response.status_code)
        return None

    data = response.json()

    items = data["tracks"]["items"]

    # NO RESULTS
    if len(items) == 0:
        return None

    track = items[0]

    # TRACK ID
    track_id = track["id"]

    # TRACK NAME
    track_name = track["name"]

    # ARTIST INFO
    artist_info = track["artists"][0]

    artist_id = artist_info["id"]
    artist_name = artist_info["name"]

    return {
        "artist_id": artist_id,
        "artist_name": artist_name,
        "song_id": track_id,
        "song_name": track_name
    }


def get_audio_features(track_id, headers):

    if pd.isna(track_id):
        return {}

    url = (
        "https://spotify-extended-audio-features-api.p.rapidapi.com"
        f"/v1/audio-features/{track_id}"
    )

    try:

        response = requests.get(
            url,
            headers=headers
        )

        time.sleep(1)

        if response.status_code == 200:

            data = response.json()

            return {

                'danceability': data.get('danceability'),

                'energy': data.get('energy'),

                'valence': data.get('valence'),

                'tempo': data.get('tempo'),

                'acousticness': data.get('acousticness'),

                'instrumentalness': data.get('instrumentalness'),

                'liveness': data.get('liveness'),

                'speechiness': data.get('speechiness')

            }

        else:

            print(
                f"AUDIO API ERROR for {track_id}: "
                f"{response.status_code}"
            )

            return {}

    except Exception as e:

        print(
            f"AUDIO REQUEST FAILED for "
            f"{track_id}: {e}"
        )

        return {}



# ==========================================
# GET ARTIST GENRES
# ==========================================

def get_artist_genres(artist_id, headers):

    if pd.isna(artist_id):
        return None

    url = (
        "https://spotify-extended-audio-features-api.p.rapidapi.com"
        f"/v1/artists/{artist_id}"
    )

    try:

        response = requests.get(
            url,
            headers=headers
        )

        time.sleep(1)

        if response.status_code == 200:

            data = response.json()

            genres = data.get("genres", [])

            # convert list → string
            genre_string = ", ".join(genres)

            return genre_string

        else:

            print(
                f"ARTIST API ERROR for "
                f"{artist_id}: {response.status_code}"
            )

            return None

    except Exception as e:

        print(
            f"ARTIST REQUEST FAILED for "
            f"{artist_id}: {e}"
        )

        return None


Overwriting /content/drive/MyDrive/music-analytics/app/spotify_api.py


In [74]:
%%writefile /content/drive/MyDrive/music-analytics/app/generate_track_artists_ids.py

from spotify_api import (
    get_token,
    get_track_and_artist_id
)

import time
import pandas as pd

from dotenv import load_dotenv
import os

load_dotenv()  # loads the .env file

CLIENT_ID = os.getenv("CLIENT_ID")
CLIENT_SECRET = os.getenv("CLIENT_SECRET")
RAPIDAPI_KEY = os.getenv("RAPIDAPI_KEY")
API_KEYS = [
    os.getenv("RAPIDAPI_KEY_1"),
    os.getenv("RAPIDAPI_KEY_2"),
    os.getenv("RAPIDAPI_KEY_3"),
    os.getenv("RAPIDAPI_KEY_4"),
    os.getenv("RAPIDAPI_KEY_5"),
    os.getenv("RAPIDAPI_KEY_6"),
    os.getenv("RAPIDAPI_KEY_7"),
    os.getenv("RAPIDAPI_KEY_8")
]

API_KEYS = [
    key for key in API_KEYS
    if key is not None
]

def enrich_track_artist_ids(df,CLIENT_ID, CLIENT_SECRET):
  # =========================================
  # GET TOKEN
  # =========================================

  token = get_token(CLIENT_ID, CLIENT_SECRET)


  # =========================================
  # CREATE NEW DATAFRAME
  # =========================================

  processed_data = []


  for index, row in df.iterrows():

      artist = row["artistName"]
      song = row["trackName"]

      print(f"Processing: {song} - {artist}")

      result = get_track_and_artist_id(
          song,
          artist,
          token
      )

      if result is not None:

          processed_data.append({

              "artist_id": result["artist_id"],

              "artist_name": result["artist_name"],

              "song_id": result["song_id"],

              "song_name": result["song_name"],

              "count": row["playCount"],

              "totalMsPlayed": row["totalMsPlayed"],
              "year_month": row["year_month"]

          })

      else:

          processed_data.append({

              "artist_id": None,

              "artist_name": artist,

              "song_id": None,

              "song_name": song,

              "count": row["playCount"],

              "totalMsPlayed": row["totalMsPlayed"],
              "year_month": row["year_month"]

          })

      # avoid rate limiting
      time.sleep(0.2)


  # =========================================
  # CREATE FINAL DATAFRAME
  # =========================================

  dtf1 = pd.DataFrame(processed_data)


  # =========================================
  # SAVE CSV
  # =========================================

  return dtf1

Overwriting /content/drive/MyDrive/music-analytics/app/generate_track_artists_ids.py


In [75]:
%%writefile /content/drive/MyDrive/music-analytics/app/generate_audio_genres_features.py

from spotify_api import (
    get_audio_features,
    get_artist_genres
)
from generate_track_artists_ids import API_KEYS

import pandas as pd
import time

def enrich_audio_and_genres(dtf2, headers):
  # headers={
  #     "x-rapidapi-key": "8814ea78a1msh30614356e5ec298p15f105jsnbbc911375a5f",
  #     "x-rapidapi-host": "spotify-extended-audio-features-api.p.rapidapi.com"
  # }

  headers = headers

  # ==========================================
  # FIND UNPROCESSED ROWS
  # ==========================================

  # if danceability OR genres missing,
  # treat as unprocessed

  unprocessed_mask = (

      (
          dtf2['danceability'].isna()
          |
          dtf2['genres'].isna()
      )

      &

      dtf2['song_id'].notna()

  )


  # ==========================================
  # TAKE NEXT 5 ROWS
  # ==========================================

  batch_to_process = dtf2[
      unprocessed_mask
  ]


  # ==========================================
  # PROCESS BATCH
  # ==========================================

  if batch_to_process.empty:

      print("ALL TRACKS PROCESSED")

  else:

      print(
          f"Processing next "
          f"{len(batch_to_process)} tracks..."
      )

      BATCH_SIZE = 5

      for batch_num, start in enumerate(
          range(0, len(batch_to_process), BATCH_SIZE)):
          batch = batch_to_process.iloc[
              start:start+BATCH_SIZE
          ]

          api_key = API_KEYS[
              batch_num % len(API_KEYS)
          ]

          headers = {
              "x-rapidapi-key": api_key,
              "x-rapidapi-host":
              "spotify-extended-audio-features-api.p.rapidapi.com"
          }

          print(
              f"\nUsing API Key "
              f"{batch_num % len(API_KEYS)+1}"
          )

          for idx, row in batch_to_process.iterrows():

              track_id = row['song_id']
              artist_id = row['artist_id']

              print(
                  f"\nProcessing:\n"
                  f"Track: {track_id}\n"
                  f"Artist: {artist_id}"
              )

              # ==================================
              # AUDIO FEATURES
              # ==================================

              audio_features = get_audio_features(
                  track_id, headers
              )

              # ==================================
              # GENRES
              # ==================================

              genres = get_artist_genres(
                  artist_id, headers
              )

              # ==================================
              # UPDATE DATAFRAME
              # ==================================

              for key, value in audio_features.items():

                  dtf2.at[idx, key] = value

              dtf2.at[idx, 'genres'] = genres

              print("DONE")

  # ==========================================
  # SAVE UPDATED CSV
  # ==========================================

  dtf2.to_csv(
      "userHistoryPreprocessed.csv",
      index=False
  )

  print(
      "\nUPDATED CSV SAVED"
  )

  return dtf2

Overwriting /content/drive/MyDrive/music-analytics/app/generate_audio_genres_features.py


In [76]:
%%writefile /content/drive/MyDrive/music-analytics/app/genre_encoding.py


def create_genre_list(dtf3):
  dtf3.dropna(inplace=True)
  dtf3['genre_list'] = dtf3.head(40)['genres'].apply(lambda x: x.split(', '))

  all_unique_genres = dtf3['genre_list'].dropna().explode().unique()
  return all_unique_genres

def one_hot_encode_genres(dtf3):
  all_unique_genres=create_genre_list(dtf3)
  for genre_name in all_unique_genres:
      print(f"Processing genre: {genre_name}")
      dtf3[genre_name] = dtf3['genre_list'].apply(lambda x: 1 if isinstance(x, list) and genre_name in x else 0)
  return dtf3

Overwriting /content/drive/MyDrive/music-analytics/app/genre_encoding.py


In [77]:
%%writefile /content/drive/MyDrive/music-analytics/app/analytics.py

import matplotlib.pyplot as plt
import numpy as np
import os

os.makedirs(
    "outputs",
    exist_ok=True
)


def genre_distribution(dtf3):

    genre_counts = dtf3.iloc[:, 17:].sum()

    genre_df = genre_counts.reset_index()

    genre_df.columns = [
        "genre",
        "frequency"
    ]

    plt.figure(figsize=(8,8))

    plt.pie(
        genre_df["frequency"],
        labels=genre_df["genre"],
        autopct="%1.1f%%"
    )

    plt.title("Genre Distribution")

    plt.savefig(
        "outputs/genre_distribution.png",
        bbox_inches="tight"
    )

    plt.close()

    return genre_df

def top_artists(dtf3):
  artist_count=dtf3.iloc[:, 1].groupby(dtf3.iloc[:, 1]).count()
  top_5_artists = artist_count.sort_values(ascending=False).head(5)
  top_5_df = top_5_artists.rename('count').reset_index()

  artist_count = (
        dtf3.groupby("artist_name")
        .size()
        .sort_values(ascending=False)
        .head(10)
    )

  plt.figure(figsize=(10,5))

  artist_count.plot(
      kind="bar"
  )

  plt.title(
      "Top Artists"
  )

  plt.ylabel(
      "Play Count"
  )

  plt.tight_layout()

  plt.savefig(
      "outputs/top_artists.png"
  )

  plt.close()

  return top_5_df

def top_tracks(df):
  top_tracks = (
    df.groupby("song_name")["count"]
        .sum()
        .sort_values(ascending=False)
        .head(5)
  )
  plt.figure(figsize=(10,5))

  tracks.plot(
      kind="bar"
  )

  plt.title(
      "Top Tracks"
  )

  plt.ylabel(
      "Play Count"
  )

  plt.tight_layout()

  plt.savefig(
      "outputs/top_tracks.png"
  )

  plt.close()
  return top_tracks


def get_last_2_months(df):
  import pandas as pd

  df["year_month"] = pd.to_datetime(df["year_month"])
  latest_month = df["year_month"].max()
  last_2_months = df[
    df["year_month"] >= (latest_month - pd.DateOffset(months=1))
  ]
  return last_2_months

def total_minutes_last_2_months(last_2_months):
  total_minutes = last_2_months["totalMsPlayed"].sum() / (1000 * 60)

  return total_minutes

def top_artists_last_2_months(last_2_months):
  top_artists = (
    last_2_months
    .groupby("artist_name")["totalMsPlayed"]
    .sum()
    .sort_values(ascending=False)
    .reset_index()
  )

  top_artists["minutes_played"] = top_artists["totalMsPlayed"] / (1000 * 60)

  # print("\nTop Artists (Last 2 Months)")
  artists = (
        last_2_months
        .groupby("artist_name")
        ["totalMsPlayed"]
        .sum()
        .sort_values(ascending=False)
        .head(10)
        / (1000*60)
    )

  plt.figure(figsize=(10,5))

  artists.plot(
      kind="bar"
  )

  plt.ylabel(
      "Minutes Played"
  )

  plt.title(
      "Top Artists (Last 2 Months)"
  )

  plt.tight_layout()

  plt.savefig(
      "outputs/top_artists_last2months.png"
  )

  plt.close()
  return top_artists[["artist_name", "minutes_played"]].head(5)

def top_tracks_last_2_months(last_2_months):

  # --------------------------------------------------
  # 3. Top Tracks (Last 2 Months)
  # --------------------------------------------------
  top_tracks = (
      last_2_months
      .groupby(["song_name", "artist_name"])["totalMsPlayed"]
      .sum()
      .sort_values(ascending=False)
      .reset_index()
  )

  top_tracks["minutes_played"] = top_tracks["totalMsPlayed"] / (1000 * 60)

  tracks = (
        last_2_months
        .groupby("song_name")
        ["totalMsPlayed"]
        .sum()
        .sort_values(ascending=False)
        .head(10)
        / (1000*60)
    )

  plt.figure(figsize=(10,5))

  tracks.plot(
      kind="bar"
  )

  plt.ylabel(
      "Minutes Played"
  )

  plt.title(
      "Top Tracks (Last 2 Months)"
  )

  plt.tight_layout()

  plt.savefig(
      "outputs/top_tracks_last2months.png"
  )

  plt.close()
  return top_tracks[["song_name", "artist_name", "minutes_played"]].head(5)


def get_audio_profile(df_last2):

  weights = df_last2["totalMsPlayed"]

  profile = {
      "danceability": (df_last2["danceability"] * weights).sum() / weights.sum(),
      "energy": (df_last2["energy"] * weights).sum() / weights.sum(),
      "valence": (df_last2["valence"] * weights).sum() / weights.sum(),
      "acousticness": (df_last2["acousticness"] * weights).sum() / weights.sum(),
      "instrumentalness": (df_last2["instrumentalness"] * weights).sum() / weights.sum(),
      "speechiness": (df_last2["speechiness"] * weights).sum() / weights.sum(),
      "tempo": (df_last2["tempo"] * weights).sum() / weights.sum()
  }

  return profile

def classify_music_mood(profile):
  energy = profile["energy"]
  valence = profile["valence"]
  danceability = profile["danceability"]
  acousticness = profile["acousticness"]
  instrumentalness = profile["instrumentalness"]

  if energy > 0.7 and valence > 0.6:
      return "Party / Energetic 🎉"

  elif energy > 0.7 and valence < 0.4:
      return "Workout / Aggressive 💪"

  elif acousticness > 0.6 and energy < 0.4:
      return "Acoustic / Relaxed 🌿"

  elif instrumentalness > 0.5:
      return "Focus / Study 📚"

  elif valence < 0.35:
      return "Melancholic 🌧️"

  elif danceability > 0.7:
      return "Dance / Groovy 🕺"

  else:
      return "Balanced Listener 🎧"

def music_mood_last_2_months(df):

  df_last2 = get_last_2_months(df)

  profile = get_audio_profile(df_last2)

  mood = classify_music_mood(profile)

  return {
      "mood": mood,
      "profile": profile
  }

def plot_music_profile(profile):
  profile_for_plot = profile.copy()

  profile_for_plot["tempo"] = (
      profile_for_plot["tempo"] / 250
  )

  labels = list(profile_for_plot.keys())

  values = list(profile_for_plot.values())

  values += values[:1]

  angles = np.linspace(
      0,
      2*np.pi,
      len(labels),
      endpoint=False
  ).tolist()

  angles += angles[:1]

  fig = plt.figure(figsize=(8,8))

  ax = plt.subplot(111, polar=True)

  ax.plot(angles, values)

  ax.fill(angles, values, alpha=0.25)

  ax.set_xticks(angles[:-1])

  ax.set_xticklabels(labels)

  plt.title("Music Profile (Last 2 Months)")

  plt.savefig(
      "outputs/music_profile_radar.png",
      bbox_inches="tight"
  )

  plt.close()


def plot_listener_mood(mood):

    plt.figure(figsize=(6,3))

    plt.text(
        0.5,
        0.5,
        mood,
        fontsize=18,
        ha="center"
    )

    plt.axis("off")

    plt.savefig(
        "outputs/listener_mood.png"
    )

    plt.close()

Overwriting /content/drive/MyDrive/music-analytics/app/analytics.py


In [78]:
# monthly_minutes = (
#     df.groupby("year_month")["totalMsPlayed"]
#       .sum()
#       / 60000
# )

# monthly_minutes.plot(kind="line")

In [79]:
# pipeline.py

In [80]:
%%writefile /content/drive/MyDrive/music-analytics/app/pipeline.py

# ==========================================
# IMPORTS
# ==========================================

import os
import pandas as pd

from dotenv import load_dotenv

# preprocessing
from preprocess import (
    create_monthly_summary
)

# spotify enrichment
from generate_track_artists_ids import (
    enrich_track_artist_ids
)

from generate_audio_genres_features import (
    enrich_audio_and_genres
)

# genre encoding
from genre_encoding import (
    one_hot_encode_genres
)

# analytics
from analytics import (
    genre_distribution,
    top_artists,
    top_tracks,
    get_last_2_months,
    total_minutes_last_2_months,
    top_artists_last_2_months,
    top_tracks_last_2_months,
    music_mood_last_2_months,
    plot_music_profile
)

# ==========================================
# LOAD ENV VARIABLES
# ==========================================

load_dotenv()

CLIENT_ID = os.getenv("CLIENT_ID")
CLIENT_SECRET = os.getenv("CLIENT_SECRET")
RAPIDAPI_KEY = os.getenv("RAPIDAPI_KEY")

# ==========================================
# RAPID API HEADERS
# ==========================================

HEADERS = {
    "x-rapidapi-key": RAPIDAPI_KEY,
    "x-rapidapi-host":
    "spotify-extended-audio-features-api.p.rapidapi.com"
}


# ==========================================
# MAIN PIPELINE
# ==========================================

def run_pipeline(json_file):

    print("\nSTEP 1 : Monthly Summary")
    print("-"*50)

    summary_df = create_monthly_summary(
        json_file
    )

    print(summary_df.head())


    # ======================================
    # STEP 2
    # ======================================

    print("\nSTEP 2 : Artist & Song IDs")
    print("-"*50)

    ids_df = enrich_track_artist_ids(
        summary_df,
        CLIENT_ID,
        CLIENT_SECRET
    )

    ids_df.to_csv(
        "track_artist_ids.csv",
        index=False
    )

    print(ids_df.head())


    # ======================================
    # STEP 3
    # ======================================

    print("\nSTEP 3 : Audio Features + Genres")
    print("-"*50)

    feature_cols = [
        "danceability",
        "energy",
        "valence",
        "tempo",
        "acousticness",
        "instrumentalness",
        "liveness",
        "speechiness",
        "genres"
    ]

    for col in feature_cols:

        if col not in ids_df.columns:
            ids_df[col] = pd.NA


    enriched_df = enrich_audio_and_genres(
        ids_df,
        HEADERS
    )

    enriched_df.to_csv(
        "spotify_enriched.csv",
        index=False
    )

    print(enriched_df.head())


    # ======================================
    # STEP 4
    # ======================================

    print("\nSTEP 4 : Genre Encoding")
    print("-"*50)

    encoded_df = one_hot_encode_genres(
        enriched_df
    )

    encoded_df.to_csv(
        "spotify_final_dataset.csv",
        index=False
    )

    print(encoded_df.shape)


    # ======================================
    # STEP 5
    # ======================================

    print("\nSTEP 5 : Analytics")
    print("-"*50)

    # Genre Pie Chart

    genre_distribution(encoded_df)


    # --------------------------------------
    # Top Artists
    # --------------------------------------

    print("\nTop Artists")

    print(
        top_artists(encoded_df)
    )


    # --------------------------------------
    # Top Tracks
    # --------------------------------------

    print("\nTop Tracks")

    print(
        top_tracks(encoded_df)
    )


    # --------------------------------------
    # Last 2 Months Stats
    # --------------------------------------

    last2 = get_last_2_months(
        encoded_df
    )

    minutes = total_minutes_last_2_months(
        last2
    )

    print(
        f"\nTotal Minutes Played "
        f"(Last 2 Months): "
        f"{minutes:.2f}"
    )


    print(
        "\nTop Artists "
        "(Last 2 Months)"
    )

    print(
        top_artists_last_2_months(
            last2
        )
    )


    print(
        "\nTop Tracks "
        "(Last 2 Months)"
    )

    print(
        top_tracks_last_2_months(
            last2
        )
    )


    # --------------------------------------
    # Music Mood
    # --------------------------------------

    mood_result = music_mood_last_2_months(
        encoded_df
    )

    print(
        "\nDetected Music Mood:"
    )

    print(
        mood_result["mood"]
    )


    print(
        "\nAudio Profile:"
    )

    print(
        mood_result["profile"]
    )


    plot_music_profile(
        mood_result["profile"]
    )

    print(
        "\nMusic profile radar chart saved."
    )

    print(
        "\nPIPELINE COMPLETED SUCCESSFULLY"
    )

    return encoded_df


# ==========================================
# RUN
# ==========================================

if __name__ == "__main__":

    JSON_FILE = "StreamingHistory_music_0.json"

    final_df = run_pipeline(
        JSON_FILE
    )

Overwriting /content/drive/MyDrive/music-analytics/app/pipeline.py


In [81]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [91]:
import os

os.chdir(
    "/content/drive/MyDrive/music-analytics/app"
)

In [92]:
!python pipeline.py


STEP 1 : Monthly Summary
--------------------------------------------------
  year_month   artistName         trackName  playCount  totalMsPlayed
0    2025-08  A.R. Rahman         Enna Sona          4         634647
1    2025-08  A.R. Rahman          Guzarish          3         653970
2    2025-08  A.R. Rahman  Jashn-E-Bahaaraa          5        1260151
3    2025-08  A.R. Rahman       Kaise Mujhe          1         343741
4    2025-08  A.R. Rahman          Maahi Ve          1         240614

STEP 2 : Artist & Song IDs
--------------------------------------------------
Status Code: 400
Response: {"error":"invalid_client","error_description":"Invalid client"}
Traceback (most recent call last):
  File "/content/drive/MyDrive/music-analytics/app/pipeline.py", line 284, in <module>
    final_df = run_pipeline(
               ^^^^^^^^^^^^^
  File "/content/drive/MyDrive/music-analytics/app/pipeline.py", line 87, in run_pipeline
    ids_df = enrich_track_artist_ids(
             ^^^^^^^^^^^^

In [84]:
from dotenv import load_dotenv
import os

load_dotenv("/content/drive/MyDrive/music-analytics/.env")

print(repr(os.getenv("CLIENT_ID")))
print(repr(os.getenv("CLIENT_SECRET")))

None
None
